# Differential Equations — Session 1  
## Section 1.1: Definitions, Classification, and Solutions

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture notebook

### Learning objectives
By the end of this session, students should be able to:

1. Distinguish an ODE from a PDE.
2. Determine the order of a differential equation.
3. Decide whether an ODE is linear or nonlinear.
4. Recognize normal form and differential form.
5. Verify explicit and implicit solutions.
6. Identify an interval on which a formula is a valid solution.
7. Interpret families, particular solutions, singular solutions, and systems.

> This notebook follows the main concepts of the publisher's Section 1.1 but uses original explanations, examples, figures, and numerical values.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Activity |
|---:|---|
| 0–10 min | Motivation and vocabulary |
| 10–30 min | Type, order, normal form, and linearity |
| 30–48 min | Verifying solutions symbolically |
| 48–63 min | Intervals of definition and implicit solutions |
| 63–78 min | Families and singular solutions |
| 78–87 min | Systems and phase-plane preview |
| 87–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import display
from scipy.integrate import solve_ivp

try:
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=4, suppress=True)

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## 1. What is a differential equation?

A **differential equation** relates an unknown function to one or more of its derivatives.

Examples:

$$
y' = 4x-y,
\qquad
y''+3y'-2y=\sin x,
\qquad
u_t=2u_{xx}.
$$

- An **ordinary differential equation (ODE)** uses ordinary derivatives with respect to one independent variable.
- A **partial differential equation (PDE)** uses partial derivatives and usually has two or more independent variables.

### Visual interpretation

A derivative describes a local rate of change. A differential equation prescribes how that rate must depend on the current state, time, or other variables.

### Classroom Checkpoint — Classify Before Calculating

Consider

$$
(1+x^2)y''+xy'-4y=e^x.
$$

Before continuing, try to identify its order, whether it is linear, and the largest real interval on which it is in standard linear form.

> Pause here. Before continuing, try to explain their reasoning before continuing.

In [ ]:
# A visual reminder: derivative = local slope
x = np.linspace(-2.5, 2.5, 400)
y = x**3 - 2*x
x0 = 1.0
y0 = x0**3 - 2*x0
slope = 3*x0**2 - 2

tangent = y0 + slope*(x - x0)

plt.plot(x, y, label=r"$y=x^3-2x$")
plt.plot(x, tangent, "--", label=f"Tangent at x={x0:g}; slope={slope:g}")
plt.scatter([x0], [y0], s=70)
plt.xlabel("x")
plt.ylabel("y")
plt.title("A derivative gives the slope of the tangent line")
plt.legend()
plt.show()

## 2. Classification checklist

For each equation, ask:

1. **Type:** ODE or PDE?
2. **Order:** What is the highest derivative?
3. **Linearity:** Are $y,y',\ldots,y^{(n)}$ present only to the first power, not multiplied together, and with coefficients depending only on the independent variable?
4. **Normal form:** Has the highest derivative been isolated?

### Linear $n$th-order ODE

$$
a_n(x)y^{(n)}+a_{n-1}(x)y^{(n-1)}+\cdots+a_1(x)y'+a_0(x)y=g(x),
\qquad a_n(x)\ne 0.
$$

The coefficients may depend on $x$, but not on $y$ or its derivatives.

### In-class classification

Classify each equation before continuing.

| Equation | Type | Order | Linear? |
|---|---|---:|---|
| $y'+(1+x^2)y=e^{-x}$ | ODE | 1 | Yes |
| $y''+y\,y'=0$ | ODE | 2 | No |
| $x^2y'''+\sin(x)y=4$ | ODE | 3 | Yes, where $x\ne0$ |
| $u_t+u\,u_x=0$ | PDE | 1 | Nonlinear |
| $(y'')^2+y=0$ | ODE | 2 | No |

## 3. Differential form and normal form

A first-order equation can sometimes be written as

$$
M(x,y)\,dx+N(x,y)\,dy=0.
$$

Because $dy=y'\,dx$, the equation

$$
(2x-y)\,dx+(1+x)\,dy=0
$$

can be written as

$$
(2x-y)+(1+x)y'=0,
$$

and, where $x\ne -1$,

$$
y'=\frac{y-2x}{1+x}.
$$

## 4. Verifying an explicit solution

Consider

$$
y'+2y=x.
$$

A proposed one-parameter family is

$$
y(x)=Ce^{-2x}+\frac{x}{2}-\frac14.
$$

We verify it by differentiation and substitution.

In [ ]:
x, C = sp.symbols("x C", real=True)
y = C*sp.exp(-2*x) + x/2 - sp.Rational(1, 4)
residual = sp.simplify(sp.diff(y, x) + 2*y - x)

print("Proposed family:")
display(y)
print("Residual after substitution:")
display(residual)
print("Residual = 0 means every value of C gives a solution.")

### Solution family as a picture

Changing $C$ selects a different solution curve. The differential equation determines the family; an initial condition will later select one member.

In [ ]:
def plot_solution_family(C_selected=1.0):
    x_vals = np.linspace(-1.5, 2.5, 500)
    for c in [-3, -1.5, 0, 1.5, 3]:
        y_vals = c*np.exp(-2*x_vals) + x_vals/2 - 0.25
        plt.plot(x_vals, y_vals, alpha=0.65, label=f"C={c:g}")
    selected = C_selected*np.exp(-2*x_vals) + x_vals/2 - 0.25
    plt.plot(x_vals, selected, linewidth=3, linestyle="--",
             label=f"selected C={C_selected:g}")
    plt.ylim(-8, 8)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(r"Family of solutions of $y'+2y=x$")
    plt.legend(ncol=2)
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        plot_solution_family,
        C_selected=FloatSlider(min=-3, max=3, step=0.25, value=1)
    )
else:
    plot_solution_family(1.0)

## 5. Interval of definition

A formula can be meaningful on a large domain but may solve the ODE only on intervals where:

- the formula is differentiable, and
- every coefficient or expression in the ODE is defined.

Example:

$$
y=\frac{1}{x-2}
$$

solves

$$
(x-2)y'+y=0.
$$

The maximal solution intervals are

$$
(-\infty,2)
\qquad\text{and}\qquad
(2,\infty).
$$

In [ ]:
x_left = np.linspace(-4, 1.92, 400)
x_right = np.linspace(2.08, 8, 400)

plt.plot(x_left, 1/(x_left-2), label=r"$y=1/(x-2)$ on $(-\infty,2)$")
plt.plot(x_right, 1/(x_right-2), label=r"$y=1/(x-2)$ on $(2,\infty)$")
plt.axvline(2, linestyle="--", label="singular point x=2")
plt.ylim(-8, 8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("A solution curve cannot pass through a singular point")
plt.legend()
plt.show()

## 6. Explicit and implicit solutions

An **explicit solution** has the form $y=\phi(x)$.

An **implicit solution** is a relation $G(x,y)=0$ that defines one or more solution branches.

Consider

$$
x^2+2y^2=18.
$$

Implicit differentiation gives

$$
2x+4yy'=0
\quad\Longrightarrow\quad
y'=-\frac{x}{2y},
$$

where $y\ne0$. The relation contains an upper and a lower branch.

In [ ]:
x_vals = np.linspace(-np.sqrt(18), np.sqrt(18), 500)
inside = (18 - x_vals**2)/2
y_upper = np.sqrt(np.maximum(inside, 0))
y_lower = -y_upper

plt.plot(x_vals, y_upper, label="upper explicit branch")
plt.plot(x_vals, y_lower, label="lower explicit branch")
plt.scatter([-np.sqrt(18), np.sqrt(18)], [0, 0], marker="x", s=80,
            label="vertical-tangent endpoints")
plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Implicit curve $x^2+2y^2=18$")
plt.legend()
plt.show()

## 7. Families, particular solutions, and singular solutions

- A **one-parameter family** usually appears for a first-order ODE.
- A **two-parameter family** usually appears for a second-order ODE.
- A **particular solution** results from choosing parameter values.
- A **singular solution** is a solution that cannot be obtained by choosing constants in the displayed family.

Example:

$$
y'=3y^{2/3}.
$$

The family

$$
y=(x-C)^3
$$

solves the equation. The equilibrium solution $y=0$ also solves it, but it is not obtained by assigning a finite value to $C$.

In [ ]:
x_vals = np.linspace(-3, 3, 500)

for c in [-2, -1, 0, 1, 2]:
    plt.plot(x_vals, (x_vals-c)**3, label=f"C={c:g}")

plt.plot(x_vals, np.zeros_like(x_vals), linewidth=3, linestyle="--",
         label="singular solution y=0")
plt.ylim(-12, 12)
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Family $y=(x-C)^3$ and the singular solution")
plt.legend(ncol=2)
plt.show()

## 8. Systems of differential equations

A system contains two or more unknown functions. For example,

$$
x'=y,\qquad y'=-x.
$$

Differentiating the first equation gives $x''=-x$, so the motion is oscillatory. In the phase plane, trajectories are circles.

In [ ]:
def oscillator(t, z):
    x, y = z
    return [y, -x]

initial_states = [(1, 0), (0.7, 0.7), (2, 0), (-1, 1)]
t_span = (0, 2*np.pi)
t_eval = np.linspace(*t_span, 500)

for z0 in initial_states:
    sol = solve_ivp(oscillator, t_span, z0, t_eval=t_eval)
    plt.plot(sol.y[0], sol.y[1], label=f"(x0,y0)={z0}")
    plt.scatter([z0[0]], [z0[1]], s=40)

plt.gca().set_aspect("equal", adjustable="box")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Phase-plane preview for $x'=y,\ y'=-x$")
plt.legend()
plt.show()

## Classroom Checkpoint — Exit Check

Classify the equation

$$
(1+t^2)y''+e^t y'+\sin(t)y=t^3.
$$

Then answer:

1. What is its order?
2. Is it linear?
3. On what real interval is it in standard linear form?
4. How many arbitrary constants would you expect in its general solution?

> Pause here. Let students commit to an answer before running the next cell.